<a href="https://colab.research.google.com/github/Victor-KKKK/AI_study/blob/main/Anomaly_Transformer_Colab_250917_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Anomaly Transformer (ICLR 2022) — Colab Notebook (Fixed Dataset Unzip)

This version **extracts any `.zip` files from `data_raw/` into `dataset/`** and then creates a symlink `data → dataset` so that both `dataset/` and `data/` paths work.

**Pipeline:**
1) Check GPU & environment  
2) Clone official repo  
3) Install dependencies  
4) Download *or* upload datasets → **unzip to `dataset/`** → link `data`  
5) Run via bash scripts (recommended)  
6) (Fallback) Run `main.py` directly  
7) (Optional) Inspect saved results


In [1]:
#@title 1) Runtime check (GPU/versions)
import os, sys
print('Python:', sys.version)
try:
    import torch
    print('PyTorch:', torch.__version__)
    print('CUDA available:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('CUDA device:', torch.cuda.get_device_name(0))
except Exception as e:
    print('PyTorch not installed yet.')
    print('Info:', e)


Python: 3.12.11 (main, Jun  4 2025, 08:56:18) [GCC 11.4.0]
PyTorch: 2.8.0+cu126
CUDA available: True
CUDA device: NVIDIA A100-SXM4-40GB


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
!pwd
%cd /content/drive/MyDrive/Anomaly-Transformer-main

/content
/content/drive/MyDrive/Anomaly-Transformer-main


In [4]:
#@title Google Driver 이용시 여기서 바로 실행하면 됨.
!chmod +x scripts/*.sh || true
!sed -i 's/\r$//' scripts/*.sh || true
print('Running MSL benchmark from scripts/...')
!bash ./scripts/MSL.sh

Running MSL benchmark from scripts/...
------------ Options -------------
anormly_ratio: 1.0
batch_size: 256
data_path: dataset/MSL
dataset: MSL
input_c: 55
k: 3
lr: 0.0001
mode: train
model_save_path: checkpoints
num_epochs: 3
output_c: 55
pretrained_model: None
win_size: 100
-------------- End ----------------
test: (73729, 55)
train: (58317, 55)
test: (73729, 55)
train: (58317, 55)
test: (73729, 55)
train: (58317, 55)
test: (73729, 55)
train: (58317, 55)
======================TRAIN MODE======================
	speed: 0.1415s/iter; left time: 82.7859s
	speed: 0.1353s/iter; left time: 65.6347s
Epoch: 1 cost time: 32.444774866104126
Epoch: 1, Steps: 228 | Train Loss: -42.3252802 Vali Loss: -45.6402458 
Validation loss decreased (inf --> -45.640246).  Saving model ...
Updating learning rate to 0.0001
	speed: 0.3144s/iter; left time: 112.2479s
	speed: 0.1355s/iter; left time: 34.8149s
Epoch: 2 cost time: 30.8011212348938
Epoch: 2, Steps: 228 | Train Loss: -47.6815611 Vali Loss: -46.300675

In [2]:
#@title 여기서 부터는 다시 코드 받을때 하면 됨.
import os, shutil
REPO_URL = "https://github.com/thuml/Anomaly-Transformer"
REPO_DIR = "Anomaly-Transformer"
if os.path.exists(REPO_DIR):
    print('Repo exists → reclone for a clean state...')
    shutil.rmtree(REPO_DIR)
!git clone --depth 1 {REPO_URL}
%cd {REPO_DIR}
!ls -la


Cloning into 'Anomaly-Transformer'...
remote: Enumerating objects: 26, done.
remote: Counting objects: 100% (26/26), done.
remote: Compressing objects: 100% (25/25), done.
remote: Total 26 (delta 4), reused 7 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (26/26), 715.85 KiB | 55.06 MiB/s, done.
Resolving deltas: 100% (4/4), done.
/content/Anomaly-Transformer
total 68
drwxr-xr-x 8 root root  4096 Sep 27 08:25 .
drwxr-xr-x 1 root root  4096 Sep 27 08:25 ..
drwxr-xr-x 2 root root  4096 Sep 27 08:25 data_factory
drwxr-xr-x 8 root root  4096 Sep 27 08:25 .git
-rw-r--r-- 1 root root  1799 Sep 27 08:25 .gitignore
-rw-r--r-- 1 root root  1084 Sep 27 08:25 LICENSE
-rwxr-xr-x 1 root root  1662 Sep 27 08:25 main.py
drwxr-xr-x 2 root root  4096 Sep 27 08:25 model
drwxr-xr-x 2 root root  4096 Sep 27 08:25 pics
-rw-r--r-- 1 root root  2472 Sep 27 08:25 README.md
-rwxr-xr-x 1 root root     0 Sep 27 08:25 results.txt
drwxr-xr-x 2 root root  4096 Sep 27 08:25 scripts
-rw-r--r-- 1 root root 

In [ ]:
#@title 3) Install dependencies
!pip -q install --upgrade pip
!pip -q install numpy pandas scikit-learn gdown tqdm
import numpy as np, pandas as pd, sklearn
print('numpy', np.__version__)
print('pandas', pd.__version__)
print('sklearn', sklearn.__version__)


In [ ]:
#@title 4) Download or Upload datasets → unzip to `dataset/` → link `data`
import os, glob, zipfile, shutil
from pathlib import Path

RAW = Path('data_raw')
DST = Path('dataset')
DST.mkdir(exist_ok=True)

print('A) Try to fetch authors\' prepared datasets via gdown (folder).')
print('   If this fails due to permissions, skip and upload zips into ./data_raw manually.')
GDRIVE_FOLDER_URL = "https://drive.google.com/drive/folders/1gisthCoE-RrKJ0j3KPV7xiibhHWT9qRm?usp=sharing"
!gdown --folder -O data_raw "{GDRIVE_FOLDER_URL}" || echo 'gdown folder download may require permission; you can upload zips to ./data_raw instead.'

RAW.mkdir(exist_ok=True)
zips = list(RAW.rglob('*.zip'))
print(f'Found {len(zips)} zip file(s) under data_raw/.')
def flatten_single_nested_dir(root: Path):
    entries = list(root.iterdir())
    files = [p for p in entries if p.is_file()]
    dirs = [p for p in entries if p.is_dir()]
    if len(files) == 0 and len(dirs) == 1:
        inner = dirs[0]
        for item in inner.iterdir():
            shutil.move(str(item), str(root / item.name))
        inner.rmdir()

for zp in zips:
    base = zp.stem  # 'SMD', 'MSL', etc.
    target = DST / base
    if target.exists():
        print(f'- Removing existing folder for {base} to re-extract...')
        shutil.rmtree(target)
    target.mkdir(parents=True, exist_ok=True)
    print(f'- Extracting {zp} → {target}')
    try:
        with zipfile.ZipFile(zp, 'r') as zf:
            zf.extractall(target)
        flatten_single_nested_dir(target)
    except zipfile.BadZipFile:
        print(f'  ! Bad zip file: {zp}')

# Create/refresh symlink: data -> dataset
import pathlib
data_path = pathlib.Path('data')
if data_path.exists() and not data_path.is_symlink():
    import shutil as _sh
    _sh.rmtree(data_path)
try:
    if data_path.is_symlink():
        data_path.unlink()
    os.symlink('dataset', 'data', target_is_directory=True)
    print('Symlink created: data → dataset')
except Exception as e:
    print('Symlink failed (likely on Windows). Using copy instead. Error:', e)
    if not data_path.exists():
        shutil.copytree('dataset', 'data')

print('\nDirectory summary:')
!find dataset -maxdepth 2 -type f | sed 's/^/ - /' | head -n 40


In [ ]:
#@title 5A) Run via provided bash scripts (recommended)
!chmod +x scripts/*.sh || true
!sed -i 's/\r$//' scripts/*.sh || true
print('Running MSL benchmark from scripts/...')
!bash ./scripts/MSL.sh
